# Wikidata SPARQL Exploration

This notebook allows you to test queries against the Wikidata SPARQL endpoint to gather history about your cities.

In [ ]:
!pip install SPARQLWrapper pandas

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

def query_wikidata(sparql_query):
    sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
    sparql.setQuery(sparql_query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()
    
    parsed_results = []
    for result in results["results"]["bindings"]:
        row = {}
        for key in result:
            row[key] = result[key]["value"]
        parsed_results.append(row)
        
    if not parsed_results: return pd.DataFrame()
    return pd.DataFrame(parsed_results)

### 1. View All Existing Properties for a City

List every piece of data Wikidata actually has on record for a city. Replace `wd:Q2807` (Madrid) with the city you want.

In [ ]:
query_all_props = """
SELECT ?propertyName ?valueLabel WHERE {
  wd:Q2807 ?propUrl ?value . 
  ?property wikibase:directClaim ?propUrl .
  ?property rdfs:label ?propertyName .
  FILTER (LANG(?propertyName) = "es")
  SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
}
"""

df_props = query_wikidata(query_all_props)
pd.set_option('display.max_rows', None)
df_props

### 2. Historic Mayors and Accurate Political Parties

This query extracts the timeline of the Mayor (`P1313` -> `P39`), and pulls in every political party they belonged to alongside the start/end dates of their party memberships. 

We then use Pandas in the next cell to accurately filter out parties that they belonged to *outside* of their term as mayor.

In [15]:
# Q2807 is Madrid
query_mayors = """
SELECT ?mayorLabel ?start ?end ?partyLabel ?partyStart ?partyEnd WHERE {
  # Find the precise position item for the city
  wd:Q2807 wdt:P1313 ?position .
  
  ?mayor p:P39 ?statement .
  ?statement ps:P39 ?position .
  
  OPTIONAL { ?statement pq:P580 ?start . }
  OPTIONAL { ?statement pq:P582 ?end . }
  
  # Pull all their lifetime political parties and dates
  OPTIONAL {
    ?mayor p:P102 ?partyStmt .
    ?partyStmt ps:P102 ?party .
    OPTIONAL { ?partyStmt pq:P580 ?partyStart . }
    OPTIONAL { ?partyStmt pq:P582 ?partyEnd . }
  }
  
  SERVICE wikibase:label { bd:serviceParam wikibase:language "es". }
}
ORDER BY DESC(?start)
"""

df_mayors_raw = query_wikidata(query_mayors)
df_mayors_raw.head()

,start,mayorLabel,partyLabel,partyStart,end,partyEnd
0,2019-06-15T00:00:00Z,José Luis Martínez-Almeida,Partido Popular,1995-01-01T00:00:00Z,NaN,NaN
1,2015-06-13T00:00:00Z,Manuela Carmena,Partido Comunista de España,1965-01-01T00:00:00Z,2019-06-15T00:00:00Z,1981-01-01T00:00:00Z
2,2015-06-13T00:00:00Z,Manuela Carmena,Ahora Madrid,2015-01-01T00:00:00Z,2019-06-15T00:00:00Z,NaN
3,2011-12-27T00:00:00Z,Ana Botella,Alianza Popular,1978-01-01T00:00:00Z,2015-06-13T00:00:00Z,1989-01-01T00:00:00Z
4,2011-12-27T00:00:00Z,Ana Botella,Partido Popular,1989-01-01T00:00:00Z,2015-06-13T00:00:00Z,NaN


#### Processing Time-Overlaps in Pandas

Politicians change parties. Wikidata sends us ALL of them. Here we filter `df_mayors_raw` so that we specifically keep the party they were an active member of **on the date they took office**.

In [16]:
df = df_mayors_raw.copy()

# Convert columns to datetime. Missing dates become NaT (Not a Time)
for col in ['start', 'end', 'partyStart', 'partyEnd']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce').dt.tz_localize(None)

# Set "current" periods to a distant future date to make comparisons work
future = pd.Timestamp('2100-01-01')
if 'end' in df.columns: df['end'] = df['end'].fillna(future)
if 'partyEnd' in df.columns: df['partyEnd'] = df['partyEnd'].fillna(future)

# Set "unknown past" periods to a distant past date
past = pd.Timestamp('1800-01-01')
if 'start' in df.columns: df['start'] = df['start'].fillna(past)
if 'partyStart' in df.columns: df['partyStart'] = df['partyStart'].fillna(past)

if 'partyStart' in df.columns and 'partyEnd' in df.columns:
    # Kept if party was active exactly when they started their mayoral term
    valid_party = (df['partyStart'] <= df['start']) & (df['partyEnd'] >= df['start'])
    
    # Note: If no party info exists at all, keep the row anyway so we don't drop the mayor
    no_party = df['partyLabel'].isna()
    df = df[valid_party | no_party]

# Drop duplicates by strictly taking the first matched party during their start term
df = df.groupby(['mayorLabel', 'start', 'end'], dropna=False).first().reset_index()

# Clean up the display datetimes
df['start'] = df['start'].replace(past, pd.NaT).dt.strftime('%Y-%m-%d')
df['end'] = df['end'].replace(future, pd.NaT).dt.strftime('%Y-%m-%d')

df.sort_values(by='start', ascending=False).head(30)

,mayorLabel,start,end,partyLabel,partyStart,partyEnd
51,José Luis Martínez-Almeida,2019-06-15,NaN,Partido Popular,1995-01-01,2100-01-01
74,Manuela Carmena,2015-06-13,2019-06-15,Ahora Madrid,2015-01-01,2100-01-01
10,Ana Botella,2011-12-27,2015-06-13,Partido Popular,1989-01-01,2100-01-01
71,Manuel Cobo Vega,2011-12-22,2011-12-27,Partido Popular,1800-01-01,2100-01-01
8,Alberto Ruiz-Gallardón,2003-06-14,2011-12-22,Partido Popular,1989-01-01,2100-01-01
58,José María Álvarez del Manzano,1991-07-05,2003-06-14,Partido Popular,1800-01-01,2100-01-01
0,Agustín Rodríguez Sahagún,1989-06-29,1991-07-05,Centro Democrático y Social,1982-01-01,2100-01-01
62,Juan Barranco,1986-01-28,1989-06-29,Partido Socialista Obrero Español,1800-01-01,2100-01-01
20,Enrique Tierno Galván,1979-04-19,1986-01-19,Partido Socialista Obrero Español,1978-01-01,1986-01-01
66,Luis María Huete,1979-01-07,1979-05-15,Partido Popular,1800-01-01,2100-01-01
